In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import MinMaxScaler
from snowflake.snowpark.context import get_active_session

!pip install tensorflow
!pip install keras

In [ ]:
X_train_res_3d = np.load('X_train_res_3d.npy')
y_train_res = np.load('y_train_res.npy')

print(X_train_res_3d.shape)
print(y_train_res.shape)

In [ ]:
import tensorflow as tf

from tensorflow.keras import callbacks
early_stopping_cb = callbacks.EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=10,         # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored metric
)

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Bidirectional, Dense, Dropout
from keras.metrics import AUC, Recall
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L1L2

l1_reg = 0.00001
l2_reg = 0.0001

model = Sequential()
# Input layer
model.add(Bidirectional(LSTM(units=150, return_sequences=True, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg)), input_shape=(X_train_res_3d.shape[1], X_train_res_3d.shape[2])))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=150, return_sequences=True, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg))))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=100, return_sequences=True, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg))))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=100, return_sequences=True, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg))))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=50, return_sequences=True, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg))))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(units=50, return_sequences=False, kernel_regularizer=L1L2(l1=l1_reg, l2=l2_reg))))
model.add(Dropout(0.2))

model.add(Dense(units=3, activation='sigmoid'))

learning_rate = 0.0004
adam_optimizer = Adam(learning_rate=learning_rate)

# Compile the model
model.compile(optimizer=adam_optimizer, loss='binary_crossentropy', metrics=['accuracy', AUC(), Recall()])

model.summary()

In [ ]:
history = model.fit(
    X_train_res_3d, 
    y_train_res, 
    epochs=100, 
    batch_size=32, 
    validation_split=0.2,
    callbacks=[early_stopping_cb],
    verbose=1  # Setting verbose=1 enables the progress bar
)

In [ ]:
test_file = pd.read_csv("submission_template.csv")
display(test_file.head(5))

In [ ]:
landsat_val_features = pd.read_csv("landsat_features_validation.csv")
display(landsat_val_features.head(5))

In [ ]:
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation.csv")
display(Terraclimate_val_df.head(5))

In [ ]:
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'nir': landsat_val_features['nir'].values,
    'green': landsat_val_features['green'].values,
    'swir16': landsat_val_features['swir16'].values,
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': Terraclimate_val_df['pet'].values,
})

In [ ]:
val_data['sensor_id'] = val_data['Latitude'].astype(str) + '_' + val_data['Longitude'].astype(str)

val_data = val_data.sort_values(['sensor_id', 'Sample Date'])

# Imputing all missing data
val_data = val_data.fillna(val_data.median(numeric_only=True))

# Checking for any missing values
val_data.isna().sum()

display(val_data.head(5))
display(val_data.shape)

In [ ]:
# For Lat/Lon, we use the specific sensor's location
# (The model needs the correct coordinates even in the 'fake' rows)
input_scaler = joblib.load("input_scaler.save")
weather_cols = ['swir22','NDMI','MNDWI','pet']
val_data[weather_cols] = input_scaler.transform(val_data[weather_cols])

# (Use the median of the 3 numerical variables)
global_median = val_data[['swir22','NDMI','MNDWI','pet']].median().values

def build_backfilled_test_set(test_df, global_median, time_steps=12):
    X = []
    
    for _, group in test_df.groupby('sensor_id'):
        weather_values = group[weather_cols].values
        
        # For every single row in this sensor's data
        for i in range(1, len(weather_values) + 1):
            # Grab all data up to the current row
            current_history = weather_values[:i]
            
            # If we have less than 12 steps, backfill with the global median
            if len(current_history) < time_steps:
                needed = time_steps - len(current_history)
                filler_block = np.tile(global_median, (needed, 1))
                window = np.vstack([filler_block, current_history])
            else:
                # Take the most recent 12 steps
                window = current_history[-time_steps:]
            
            X.append(window)
            
    return np.array(X)

X_test_imputed = build_backfilled_test_set(val_data, global_median, time_steps=5)
print(f"Final Shape: {X_test_imputed.shape}")

In [ ]:
y_pred = model.predict(X_test_imputed)

display(y_pred.shape)

output_scaler = joblib.load("output_scaler.save")

final_preds = output_scaler.inverse_transform(y_pred)

test_file['sensor_id'] = test_file['Latitude'].astype(str) + '_' + test_file['Longitude'].astype(str)

test_file = test_file.sort_values(['sensor_id', 'Sample Date'])

submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': final_preds[:, 0],
    'Electrical Conductance': final_preds[:, 1],
    'Dissolved Reactive Phosphorus': final_preds[:, 2]
})

display(submission_df.head(5))